# U.S. Analytics Job Market — Scaled Data Ingestion & Cleaning

## Objective

This notebook scales the prototype API workflow into a larger multi-role dataset.

The goals are to:

- Retrieve multiple pages of job postings from the Adzuna API
- Collect Data Analyst, Business Analyst, and Product Analyst postings
- Combine and deduplicate results
- Apply a consistent cleaning pipeline
- Prepare the analytical dataset for skill extraction and BigQuery

## 1. Environment Setup

Import the libraries required for API requests, data processing, credential management, and controlled request timing.

In [1]:
import requests
import os
import time
import pandas as pd

from dotenv import load_dotenv


/Users/chrls/Desktop/求职/resume/github project/us-job-market-analytics/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
load_dotenv("../.env")

app_id = os.getenv("ADZUNA_APP_ID")
app_key = os.getenv("ADZUNA_APP_KEY")

print(app_id is not None)
print(app_key is not None)


True
True


## 2. Understand API Pagination

The API returns search results in pages. Instead of requesting only the first page, multiple page numbers can be requested sequentially and combined into one dataset.

In [3]:
page = 2

url = f"https://api.adzuna.com/v1/api/jobs/us/search/{page}"

url


'https://api.adzuna.com/v1/api/jobs/us/search/2'

In [4]:
pages_to_fetch = 5
results_per_page = 50


### 2.1 Configure the Initial Multi-Page Test

Start with five pages of Data Analyst postings to validate the pagination workflow before expanding to multiple roles.

In [5]:
search_term = "data analyst"
pages_to_fetch = 5
results_per_page = 50


In [6]:
all_jobs = []

for page in range(1, pages_to_fetch + 1):
    print(page)

1
2
3
4
5


In [7]:
for page in range(1, pages_to_fetch + 1):
    url = f"https://api.adzuna.com/v1/api/jobs/us/search/{page}"
    print(url)


https://api.adzuna.com/v1/api/jobs/us/search/1
https://api.adzuna.com/v1/api/jobs/us/search/2
https://api.adzuna.com/v1/api/jobs/us/search/3
https://api.adzuna.com/v1/api/jobs/us/search/4
https://api.adzuna.com/v1/api/jobs/us/search/5


### 2.2 Fetch Multiple Pages

Loop through page numbers, send one API request per page, and append each page's job records to a combined Python list.

In [8]:
all_jobs = []

for page in range(1, pages_to_fetch + 1):
    url = f"https://api.adzuna.com/v1/api/jobs/us/search/{page}"

    params = {
        "app_id": app_id,
        "app_key": app_key,
        "results_per_page": results_per_page,
        "what": search_term,
    }

    response = requests.get(url, params=params)
    response.raise_for_status()

    data = response.json()
    page_jobs = data["results"]

    print(f"Page {page}: {len(page_jobs)} jobs")

    all_jobs.extend(page_jobs)

    time.sleep(0.2)


Page 1: 50 jobs
Page 2: 50 jobs
Page 3: 50 jobs
Page 4: 50 jobs
Page 5: 50 jobs


In [9]:
len(all_jobs)


250

In [10]:
type(all_jobs)


list

### 2.3 Combine Multi-Page Results

Flatten the combined JSON job records into a single Pandas DataFrame for validation.

In [11]:
da_df = pd.json_normalize(all_jobs)


In [12]:
da_df.shape


(250, 21)

In [13]:
da_df.head()


,id,description,created,redirect_url,latitude,salary_max,title,longitude,salary_min,salary_is_predicted,...,adref,category.tag,category.label,category.__CLASS__,company.display_name,company.__CLASS__,location.area,location.display_name,location.__CLASS__,contract_time
0,5901208184,Job Family : Data Science Consulting Travel Re...,2026-09-28T00:37:44Z,https://www.adzuna.com/land/ad/5901208184?se=V...,34.761179,103253.55,Data Analyst,-86.686621,103253.55,1,...,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTkwMTIwODE4NCIsI...,it-jobs,IT Jobs,Adzuna::API::Response::Category,Guidehouse,Adzuna::API::Response::Company,"[US, Alabama, Madison County, Madison]","Madison, Madison County",Adzuna::API::Response::Location,NaN
1,5896874781,"Data Analyst Location: Hybrid | Nashville, TN ...",2026-09-24T08:55:11Z,https://www.adzuna.com/land/ad/5896874781?se=V...,36.172489,120640.00,Data analyst,-86.780596,120640.00,0,...,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTg5Njg3NDc4MSIsI...,it-jobs,IT Jobs,Adzuna::API::Response::Category,Vaco LLC,Adzuna::API::Response::Company,"[US, Tennessee, Davidson County, Nashville]","Nashville, Davidson County",Adzuna::API::Response::Location,NaN
2,5886631734,Arnold & Porter is an international law firm w...,2026-09-16T16:12:22Z,https://www.adzuna.com/land/ad/5886631734?se=V...,38.912068,110000.00,Human Resources Data Analyst,-77.019023,82400.00,0,...,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTg4NjYzMTczNCIsI...,hr-jobs,HR & Recruitment Jobs,Adzuna::API::Response::Category,Arnold Porter,Adzuna::API::Response::Company,"[US, Washington, D.C., Washington]","Washington, Washington, D.C.",Adzuna::API::Response::Location,full_time
3,5886391534,"Hi, We currently have an exciting opportunity ...",2026-09-16T12:07:46Z,https://www.adzuna.com/land/ad/5886391534?se=V...,38.893471,65915.28,Data Analyst (Financials data Analyst),-77.220397,65915.28,1,...,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTg4NjM5MTUzNCIsI...,accounting-finance-jobs,Accounting & Finance Jobs,Adzuna::API::Response::Category,"eSolutionsFirst, LLC",Adzuna::API::Response::Company,"[US, Virginia, Fairfax County, Vienna]","Vienna, Fairfax County",Adzuna::API::Response::Location,NaN
4,5888579132,hackajob is collaborating with MANTECH to conn...,2026-09-17T22:33:05Z,https://www.adzuna.com/land/ad/5888579132?se=V...,32.372413,78684.16,Data Analyst,-81.311499,78684.16,1,...,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTg4ODU3OTEzMiIsI...,it-jobs,IT Jobs,Adzuna::API::Response::Category,MANTECH,Adzuna::API::Response::Company,"[US, Georgia, Effingham County, Springfield]","Springfield, Effingham County",Adzuna::API::Response::Location,NaN


In [14]:
da_df["id"].nunique()


250

### 2.4 Build a Reusable Job Ingestion Function

Convert the multi-page ingestion workflow into a reusable function so the same logic can be applied consistently across multiple analytics roles without duplicating code.

In [15]:
def fetch_jobs(search_term, pages=5, results_per_page=50):
    all_jobs = []

    for page in range(1, pages + 1):
        url = f"https://api.adzuna.com/v1/api/jobs/us/search/{page}"

        params = {
            "app_id": app_id,
            "app_key": app_key,
            "results_per_page": results_per_page,
            "what": search_term,
        }

        response = requests.get(url, params=params, timeout=30)

        response.raise_for_status()

        data = response.json()
        page_jobs = data.get("results", [])

        print(f"{search_term} | Page {page}: {len(page_jobs)} jobs")

        all_jobs.extend(page_jobs)

        time.sleep(0.2)

    jobs_df = pd.json_normalize(all_jobs)

    jobs_df["search_role"] = search_term

    return jobs_df


### 2.5 Retrieve Multiple Analytics Roles

Use the reusable ingestion function to collect job postings for Data Analyst, Business Analyst, and Product Analyst searches.

In [16]:
da_df = fetch_jobs("data analyst")


data analyst | Page 1: 50 jobs
data analyst | Page 2: 50 jobs
data analyst | Page 3: 50 jobs
data analyst | Page 4: 50 jobs
data analyst | Page 5: 50 jobs


In [17]:
ba_df = fetch_jobs("business analyst")


business analyst | Page 1: 50 jobs
business analyst | Page 2: 50 jobs
business analyst | Page 3: 50 jobs
business analyst | Page 4: 50 jobs
business analyst | Page 5: 50 jobs


In [18]:
pa_df = fetch_jobs("product analyst")


product analyst | Page 1: 50 jobs
product analyst | Page 2: 50 jobs
product analyst | Page 3: 50 jobs
product analyst | Page 4: 50 jobs
product analyst | Page 5: 50 jobs


In [19]:
print("Data Analyst:", da_df.shape)
print("Business Analyst:", ba_df.shape)
print("Product Analyst:", pa_df.shape)


Data Analyst: (250, 22)
Business Analyst: (250, 21)
Product Analyst: (250, 23)


### 2.6 Combine Role-Level Datasets

Combine the three role-level DataFrames into one dataset while preserving the original search role for each record.

In [20]:
combined_df = pd.concat([da_df, ba_df, pa_df], ignore_index=True)


In [21]:
combined_df.shape


(750, 23)

In [22]:
combined_df["search_role"].value_counts()


search_role
data analyst        250
business analyst    250
product analyst     250
Name: count, dtype: int64

In [23]:
print("Total rows:", len(combined_df))
print("Unique job IDs:", combined_df["id"].nunique())


Total rows: 750
Unique job IDs: 750


In [24]:
duplicate_jobs = combined_df[
    combined_df.duplicated(subset="id", keep=False)
].sort_values("id")


In [25]:
duplicate_jobs[["id", "title", "company.display_name", "search_role"]].head(20)


,id,title,company.display_name,search_role


### 2.7 Validate the Combined Dataset

Before applying the cleaning pipeline, inspect the combined multi-role dataset for schema differences, missing values, duplicates, and location hierarchy variation.

In [26]:
combined_df.shape


(750, 23)

In [27]:
combined_df.columns.tolist()


['salary_max',
 'salary_min',
 'longitude',
 'title',
 'salary_is_predicted',
 '__CLASS__',
 'adref',
 'id',
 'description',
 'created',
 'redirect_url',
 'latitude',
 'location.area',
 'location.display_name',
 'location.__CLASS__',
 'category.label',
 'category.tag',
 'category.__CLASS__',
 'company.display_name',
 'company.__CLASS__',
 'contract_time',
 'search_role',
 'contract_type']

In [28]:
missing_summary = pd.DataFrame(
    {
        "missing_count": combined_df.isna().sum(),
        "missing_pct": (combined_df.isna().mean() * 100).round(1),
    }
).sort_values("missing_pct", ascending=False)

missing_summary


,missing_count,missing_pct
contract_type,743,99.1
contract_time,712,94.9
latitude,81,10.8
longitude,81,10.8
title,0,0.0
salary_is_predicted,0,0.0
search_role,0,0.0
company.__CLASS__,0,0.0
company.display_name,0,0.0
category.__CLASS__,0,0.0


### 2.8 Validate Location Hierarchies at Scale

Re-check location hierarchy lengths on the larger dataset before applying geographic parsing logic. The larger sample may contain structures not observed in the initial 10-record prototype.

In [29]:
combined_df["location.area"].apply(
    lambda x: len(x) if isinstance(x, list) else None
).value_counts(dropna=False).sort_index()


location.area
1     77
2     17
3     15
4    611
5     30
Name: count, dtype: int64

In [30]:
combined_df["location_depth"] = combined_df["location.area"].apply(
    lambda x: len(x) if isinstance(x, list) else None
)


In [31]:
combined_df[["location.display_name", "location.area", "location_depth"]].sort_values(
    "location_depth"
).head(20)


,location.display_name,location.area,location_depth
510,US,[US],1
215,US,[US],1
218,US,[US],1
219,US,[US],1
238,US,[US],1
527,US,[US],1
247,US,[US],1
517,US,[US],1
512,US,[US],1
259,US,[US],1


### 2.9 Select Analytical Fields

Retain fields required for market, location, salary, skill, and entry-level analysis while preserving the search query that produced each record.

In [32]:
analysis_columns = [
    "id",
    "created",
    "title",
    "description",
    "company.display_name",
    "location.area",
    "location.display_name",
    "latitude",
    "longitude",
    "salary_min",
    "salary_max",
    "salary_is_predicted",
    "search_role",
]


In [33]:
jobs_df = combined_df[analysis_columns].copy()


In [34]:
jobs_df.shape


(750, 13)

### 2.10 Standardize Column Names

Rename flattened JSON fields to simpler analytical column names that are easier to use in Python, SQL, and Tableau.

In [35]:
jobs_df = jobs_df.rename(
    columns={
        "company.display_name": "company",
        "location.display_name": "location",
        "location.area": "location_area",
    }
)


In [36]:
jobs_df.columns.tolist()


['id',
 'created',
 'title',
 'description',
 'company',
 'location_area',
 'location',
 'latitude',
 'longitude',
 'salary_min',
 'salary_max',
 'salary_is_predicted',
 'search_role']

### 2.11 Standardize Data Types

Convert timestamps and indicator fields into consistent analytical data types.

In [37]:
jobs_df["created"] = pd.to_datetime(jobs_df["created"], utc=True, errors="coerce")


In [38]:
jobs_df["salary_is_predicted"] = pd.to_numeric(
    jobs_df["salary_is_predicted"], errors="coerce"
)


In [39]:
jobs_df[["created", "salary_is_predicted"]].dtypes


created                datetime64[ns, UTC]
salary_is_predicted                  int64
dtype: object

### 2.12 Validate Salary Fields

Evaluate salary coverage and validate reported ranges before constructing compensation metrics.

In [40]:
jobs_df[["salary_min", "salary_max"]].isna().sum()


salary_min    0
salary_max    0
dtype: int64

In [41]:
print("Invalid salary ranges:", (jobs_df["salary_min"] > jobs_df["salary_max"]).sum())

print("Non-positive salary_min:", (jobs_df["salary_min"] <= 0).sum())

print("Non-positive salary_max:", (jobs_df["salary_max"] <= 0).sum())


Invalid salary ranges: 0
Non-positive salary_min: 0
Non-positive salary_max: 0


In [42]:
jobs_df["salary_midpoint"] = (jobs_df["salary_min"] + jobs_df["salary_max"]) / 2


### 2.13 Salary Prediction Coverage

Evaluate how much of the salary data is based on Adzuna-predicted values versus non-predicted values before using compensation metrics for business analysis.

In [43]:
jobs_df["salary_is_predicted"].value_counts(dropna=False)


salary_is_predicted
1    667
0     83
Name: count, dtype: int64

In [44]:
(
    jobs_df["salary_is_predicted"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(1)
)


salary_is_predicted
1    88.9
0    11.1
Name: proportion, dtype: float64

In [45]:
pd.crosstab(
    jobs_df["search_role"], jobs_df["salary_is_predicted"], normalize="index"
).mul(100).round(1)


salary_is_predicted,0,1
search_role,,
business analyst,0.8,99.2
data analyst,17.2,82.8
product analyst,15.2,84.8


#### Salary Quality Observation

Salary coverage is complete in the collected sample, but 88.9% of salary values are flagged as predicted by Adzuna.

Therefore, salary analysis should distinguish between predicted and non-predicted compensation values rather than treating all salary records as employer-reported salaries.

In [46]:
combined_df["location_depth"].value_counts(dropna=False).sort_index()


location_depth
1     77
2     17
3     15
4    611
5     30
Name: count, dtype: int64

In [47]:
combined_df[["location.display_name", "location.area", "location_depth"]].groupby(
    "location_depth", group_keys=False
).head(3)


,location.display_name,location.area,location_depth
0,"Madison, Madison County","[US, Alabama, Madison County, Madison]",4
1,"Nashville, Davidson County","[US, Tennessee, Davidson County, Nashville]",4
2,"Washington, Washington, D.C.","[US, Washington, D.C., Washington]",3
3,"Vienna, Fairfax County","[US, Virginia, Fairfax County, Vienna]",4
6,"Denver, Colorado","[US, Colorado, Denver]",3
8,US,[US],1
13,US,[US],1
14,"Grand Central, Manhattan","[US, New York, New York City, Manhattan, Grand...",5
21,"Connecticut, US","[US, Connecticut]",2
23,"Pennsylvania, US","[US, Pennsylvania]",2


### 2.14 Geographic Parsing Strategy

The larger dataset contains location hierarchies ranging from one to five levels.

Because the hierarchy is not structurally consistent across U.S. locations, only stable geographic fields are extracted programmatically. State-level geography will be used for aggregated analysis, while the original location name and coordinates are retained for more granular visualization.

In [48]:
jobs_df["country"] = jobs_df["location_area"].apply(
    lambda x: x[0] if isinstance(x, list) and len(x) >= 1 else None
)


In [49]:
jobs_df["state"] = jobs_df["location_area"].apply(
    lambda x: x[1] if isinstance(x, list) and len(x) >= 2 else None
)


In [50]:
jobs_df["location_depth"] = jobs_df["location_area"].apply(
    lambda x: len(x) if isinstance(x, list) else None
)


In [51]:
jobs_df[["location", "location_area", "country", "state", "location_depth"]].head(20)


,location,location_area,country,state,location_depth
0,"Madison, Madison County","[US, Alabama, Madison County, Madison]",US,Alabama,4
1,"Nashville, Davidson County","[US, Tennessee, Davidson County, Nashville]",US,Tennessee,4
2,"Washington, Washington, D.C.","[US, Washington, D.C., Washington]",US,"Washington, D.C.",3
3,"Vienna, Fairfax County","[US, Virginia, Fairfax County, Vienna]",US,Virginia,4
4,"Springfield, Effingham County","[US, Georgia, Effingham County, Springfield]",US,Georgia,4
5,"Tampa, Hillsborough County","[US, Florida, Hillsborough County, Tampa]",US,Florida,4
6,"Denver, Colorado","[US, Colorado, Denver]",US,Colorado,3
7,"Rosslyn, Arlington County","[US, Virginia, Arlington County, Rosslyn]",US,Virginia,4
8,US,[US],US,None,1
9,"Salt Lake City, Salt Lake County","[US, Utah, Salt Lake County, Salt Lake City]",US,Utah,4


In [52]:
jobs_df["state"].isna().sum()


np.int64(77)

In [53]:
state_coverage = jobs_df["state"].notna().mean() * 100

round(state_coverage, 1)


np.float64(89.7)

In [54]:
jobs_df["salary_source"] = jobs_df["salary_is_predicted"].map(
    {0: "non_predicted", 1: "predicted"}
)


In [55]:
jobs_df[
    ["search_role", "salary_min", "salary_max", "salary_midpoint", "salary_source"]
].head()


,search_role,salary_min,salary_max,salary_midpoint,salary_source
0,data analyst,103253.55,103253.55,103253.55,predicted
1,data analyst,120640.00,120640.00,120640.00,non_predicted
2,data analyst,82400.00,110000.00,96200.00,non_predicted
3,data analyst,65915.28,65915.28,65915.28,predicted
4,data analyst,78684.16,78684.16,78684.16,predicted


#### Geographic Quality Observation

The current 750-record ingestion run contains location hierarchies ranging from one to five levels:

- 77 records contain country-level geography only
- 17 records contain country and state
- 15 records contain three geographic levels
- 611 records contain four geographic levels
- 30 records contain five geographic levels

State is available for 89.7% of records. Because lower-level geography is not structurally consistent, the processed dataset keeps stable state-level geography, the original display location, coordinates, and a flattened location hierarchy rather than forcing every record into a fixed county/city schema.

### 2.15 Standardize Text and Traceability Fields

Standardize whitespace in core text fields and create flat traceability fields for downstream SQL and Tableau use.

In [56]:
for col in ["title", "company", "location"]:
    jobs_df[col] = jobs_df[col].astype("string").str.strip()

jobs_df["location_hierarchy"] = jobs_df["location_area"].apply(
    lambda x: " > ".join(x) if isinstance(x, list) else None
)

jobs_df["data_source"] = "Adzuna API"
jobs_df["ingested_at_utc"] = pd.Timestamp.now(tz="UTC")

In [57]:
text_quality = pd.DataFrame(
    {
        "null_count": [
            jobs_df["title"].isna().sum(),
            jobs_df["description"].isna().sum(),
            jobs_df["company"].isna().sum(),
        ],
        "blank_count": [
            jobs_df["title"].fillna("").str.strip().eq("").sum(),
            jobs_df["description"].fillna("").str.strip().eq("").sum(),
            jobs_df["company"].fillna("").str.strip().eq("").sum(),
        ],
    },
    index=["title", "description", "company"],
)

text_quality

,null_count,blank_count
title,0,0
description,0,0
company,0,0


### 2.16 Preserve Search-Role Matches and Deduplicate

A job can theoretically appear under more than one search query. Preserve all matched search roles before deduplicating on the Adzuna job ID.

In the current ingestion run, all 750 job IDs are unique, so this step does not remove any records. The logic is retained to make future runs more robust.

In [58]:
jobs_df["matched_search_roles"] = jobs_df.groupby("id")["search_role"].transform(
    lambda s: ", ".join(sorted(set(s.dropna())))
)

rows_before_dedup = len(jobs_df)

jobs_df = jobs_df.drop_duplicates(subset="id", keep="first").copy()

rows_after_dedup = len(jobs_df)

print("Rows before deduplication:", rows_before_dedup)
print("Rows after deduplication:", rows_after_dedup)
print("Rows removed:", rows_before_dedup - rows_after_dedup)

Rows before deduplication: 750
Rows after deduplication: 750
Rows removed: 0


### 2.17 Build the Final Analytical Dataset

Create a flat analytical table containing the fields required for market, skill, salary, and location analysis. The raw API response remains conceptually separate from this processed analytical layer.

In [59]:
final_columns = [
    "id",
    "created",
    "ingested_at_utc",
    "search_role",
    "matched_search_roles",
    "title",
    "company",
    "description",
    "country",
    "state",
    "location",
    "location_hierarchy",
    "location_depth",
    "latitude",
    "longitude",
    "salary_min",
    "salary_max",
    "salary_midpoint",
    "salary_is_predicted",
    "salary_source",
    "data_source",
]

final_df = jobs_df[final_columns].copy()

final_df.shape

(750, 21)

In [60]:
final_df.head()

,id,created,ingested_at_utc,search_role,matched_search_roles,title,company,description,country,state,...,location_hierarchy,location_depth,latitude,longitude,salary_min,salary_max,salary_midpoint,salary_is_predicted,salary_source,data_source
0,5901208184,2026-09-28 00:37:44+00:00,2026-09-29 01:43:24.846950+00:00,data analyst,data analyst,Data Analyst,Guidehouse,Job Family : Data Science Consulting Travel Re...,US,Alabama,...,US > Alabama > Madison County > Madison,4,34.761179,-86.686621,103253.55,103253.55,103253.55,1,predicted,Adzuna API
1,5896874781,2026-09-24 08:55:11+00:00,2026-09-29 01:43:24.846950+00:00,data analyst,data analyst,Data analyst,Vaco LLC,"Data Analyst Location: Hybrid | Nashville, TN ...",US,Tennessee,...,US > Tennessee > Davidson County > Nashville,4,36.172489,-86.780596,120640.00,120640.00,120640.00,0,non_predicted,Adzuna API
2,5886631734,2026-09-16 16:12:22+00:00,2026-09-29 01:43:24.846950+00:00,data analyst,data analyst,Human Resources Data Analyst,Arnold Porter,Arnold & Porter is an international law firm w...,US,"Washington, D.C.",...,"US > Washington, D.C. > Washington",3,38.912068,-77.019023,82400.00,110000.00,96200.00,0,non_predicted,Adzuna API
3,5886391534,2026-09-16 12:07:46+00:00,2026-09-29 01:43:24.846950+00:00,data analyst,data analyst,Data Analyst (Financials data Analyst),"eSolutionsFirst, LLC","Hi, We currently have an exciting opportunity ...",US,Virginia,...,US > Virginia > Fairfax County > Vienna,4,38.893471,-77.220397,65915.28,65915.28,65915.28,1,predicted,Adzuna API
4,5888579132,2026-09-17 22:33:05+00:00,2026-09-29 01:43:24.846950+00:00,data analyst,data analyst,Data Analyst,MANTECH,hackajob is collaborating with MANTECH to conn...,US,Georgia,...,US > Georgia > Effingham County > Springfield,4,32.372413,-81.311499,78684.16,78684.16,78684.16,1,predicted,Adzuna API


### 2.18 Final Validation

Run a final set of quality checks before exporting the processed snapshot.

In [61]:
final_quality_checks = pd.Series(
    {
        "rows": len(final_df),
        "unique_job_ids": final_df["id"].nunique(),
        "duplicate_job_ids": final_df["id"].duplicated().sum(),
        "missing_titles": final_df["title"].isna().sum(),
        "missing_descriptions": final_df["description"].isna().sum(),
        "missing_states": final_df["state"].isna().sum(),
        "missing_latitude": final_df["latitude"].isna().sum(),
        "missing_longitude": final_df["longitude"].isna().sum(),
        "invalid_salary_ranges": (
            final_df["salary_min"] > final_df["salary_max"]
        ).sum(),
        "non_positive_salary_min": (final_df["salary_min"] <= 0).sum(),
        "non_positive_salary_max": (final_df["salary_max"] <= 0).sum(),
    }
)

final_quality_checks

rows                       750
unique_job_ids             750
duplicate_job_ids            0
missing_titles               0
missing_descriptions         0
missing_states              77
missing_latitude            81
missing_longitude           81
invalid_salary_ranges        0
non_positive_salary_min      0
non_positive_salary_max      0
dtype: int64

In [62]:
salary_source_summary = (
    final_df["salary_source"]
    .value_counts(dropna=False)
    .rename_axis("salary_source")
    .to_frame("job_count")
)

salary_source_summary["share_pct"] = (
    salary_source_summary["job_count"] / len(final_df) * 100
).round(1)

salary_source_summary

,job_count,share_pct
salary_source,,
predicted,667,88.9
non_predicted,83,11.1


### 2.19 Export the Processed Snapshot

Save the cleaned analytical dataset locally in both Parquet and CSV formats.

The `data/processed/` directory is excluded from Git version control, so the full job-level dataset remains local while the reproducible ingestion and cleaning logic is published on GitHub.

In [67]:
from pathlib import Path

processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

snapshot_date = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d")

parquet_path = processed_dir / f"us_analytics_jobs_{snapshot_date}.parquet"
csv_path = processed_dir / f"us_analytics_jobs_{snapshot_date}.csv"

final_df.to_parquet(parquet_path, index=False, engine="pyarrow")
final_df.to_csv(csv_path, index=False)

print("Parquet:", parquet_path)
print("CSV:", csv_path)

Parquet: ../data/processed/us_analytics_jobs_2026-09-29.parquet
CSV: ../data/processed/us_analytics_jobs_2026-09-29.csv


## 3. Summary

This notebook scales the initial API prototype into a reproducible multi-role ingestion and cleaning workflow.

### Current ingestion run

- 750 total job records were collected
- 250 records were retrieved for each search query:
  - Data Analyst
  - Business Analyst
  - Product Analyst
- All 750 Adzuna job IDs were unique in this run
- Salary fields were complete and contained no non-positive or inverted ranges
- 88.9% of salary values were flagged as Adzuna-predicted and 11.1% as non-predicted
- State-level geography was available for 89.7% of records
- Location hierarchies varied from one to five levels, so the pipeline retains stable state-level geography plus original location information instead of forcing a universal county/city structure

### Analytical output

The processed dataset now contains:

- Job identity and posting timestamp
- Search-role provenance
- Job title, company, and description
- State-level and original geographic information
- Latitude and longitude when available
- Salary minimum, maximum, midpoint, and salary-source quality flag
- Ingestion timestamp and data-source attribution

### Next step

The next notebook will focus on skill extraction and entry-level classification from job titles and descriptions. Business analysis will then move into BigQuery and SQL, followed by Tableau visualization.

Data source: Adzuna API.